In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util


# ==============================================================================
# 1. Global parameters & He atom definition
# ==============================================================================
# He atom: 2 electrons, single atom system
geometry = [('He', (0., 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("He atom FCI reference energy")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  Excitation: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
# He atom: STO-3G basis gives 1 orbital (1s), 2 electrons
# Fermionic Hilbert space: n_orbitals=1, n_fermions_per_spin=(1,1)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=1,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. Neural network Ansatz
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

# ==============================================================================
# 3. Initialize model, sampler, optimizer
# ==============================================================================
model = SingleStateAnsatz(2, 12, rngs=nnx.Rngs(21))  # 2 spin orbitals
# Sampler (He is single atom, no graph structure needed)
sampler = nk.sampler.MetropolisSampler(hi, rule=nk.sampler.rules.Exchange(), n_chains=100, sweep_size=32)

optimizer = nk.optimizer.Sgd(learning_rate=0.1)

vstate = nk.vqs.MCState(sampler, model, n_samples=1008)

gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate,
    preconditioner=nk.optimizer.SR(diag_shift=0.1, holomorphic=True),
)

In [ ]:
# ===================== 4. Wrap model as machine function =====================
def create_machine(model: nnx.Module):
    """Wrap Flax NNX model into NetKet-style machine function"""
    graphdef, state = nnx.split(model)
    
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    
    return machine, graphdef, state

# ===================== 5. Pure JAX force-based gradient computation =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    Compute local energy E_loc(sigma) = sum_eta H(sigma->eta) psi(eta)/psi(sigma)
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)


def statistics(x):
    """Compute sample statistics"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    Core: replicate NetKet's forces_expect_hermitian function
    
    Use force-based gradient computation:
    grad E = <(E_loc - <E>) grad log psi>
    """
    # 1. Compute local energy
    O_loc = compute_local_energies(machine, params, sigma)
    
    # 2. Compute energy statistics
    O_mean, O_std = statistics(O_loc)
    
    # 3. Center local energy
    O_centered = O_loc - O_mean
    
    # 4. Compute grad log psi for each sample
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # 5. Compute force-based gradient
    def weight_and_mean(grad_component):
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    
    return O_mean, O_std, grad


@partial(jax.jit, static_argnames=("machine",))
def compute_qgt(machine, params, sigma, diag_shift=0.1):
    """
    Compute Quantum Geometric Tensor (QGT) / F matrix
    
    QGT definition:
    S_ij = <d_i log psi* d_j log psi> - <d_i log psi*><d_j log psi>
    """
    n_samples = sigma.shape[0]
    
    # Step 1: Compute grad log psi for each sample
    def log_psi_single(p, s):
        return machine(p, s)
    
    def compute_grad_for_sample(s):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=True)(params)
    
    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)
    
    # Step 2: Flatten PyTree to matrix
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)
    
    # Step 3: Center
    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean
    
    # Step 4: Compute QGT
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    
    # Step 5: Add regularization
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    
    return qgt_reg, unravel_fn

In [ ]:
# ===================== 6. Initialization =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(2, hidden_dim=12, rngs=rngs)  # 2 spin orbitals
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# Training parameters
N_ITER = 300
N_SAMPLES = 1008

# ===================== 7. Training loop =====================
print("\n" + "="*60)
print("Starting pure JAX VMC training (He atom - Natural Gradient)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

for step in range(N_ITER):
    # 1. Sampling
    sampler_state = sampler.reset(machine, params, sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. Compute force-based energy and gradient
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x * 2, grad)
    
    # 3. Compute QGT and natural gradient
    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001)
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
    
    # Natural gradient = S^{-1} * grad
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
    
    # 4. Update parameters
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. Record history
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} +/- {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# Final results
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"Training complete!")
print(f"Final energy: {final_energy.real:.8f} +/- {final_std:.6f} Ha")
print(f"FCI reference: {E_fcis[0]:.8f} Ha")
print(f"Absolute error: {final_error:.6f} Ha")
print(f"Relative error: {final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
# ===================== 8. Save model state =====================
_, state = nnx.split(model)

import orbax.checkpoint as ocp
import os
ckpt_dir = ocp.test_utils.erase_and_create_empty('./He_checkpoint')
checkpointer = ocp.StandardCheckpointer()
checkpointer.save(ckpt_dir / 'ground_state', state)